# Chapter 11 - HMM Exercise 1: Weather and Activities

Nguyễn Phúc Minh Châu - 22302421

### Task Description
A tourist is traveling in a city for **4 days**. Each day, the tourist performs one activity, but you cannot directly observe the weather. You only observe the activity.

You model this with a Hidden Markov Model (HMM):

**Hidden states (weather):**
- `sunny`
- `rainy`

**Observations (activities):**
- `walk`
- `shop`
- `museum`

**Initial Probabilities:**
- P(sunny) = 0.6
- P(rainy) = 0.4

**Transition Probabilities:**
| From → To | sunny | rainy |
|-----------|--------|--------|
| sunny     | 0.7    | 0.3    |
| rainy     | 0.4    | 0.6    |

**Emission Probabilities:**
| Weather → Activity | walk | shop | museum |
|-------------------|------|-------|---------|
| sunny             | 0.6  | 0.3   | 0.1     |
| rainy             | 0.1  | 0.4   | 0.5     |

### Observed sequence (4 days):
`O = ["walk", "shop", "museum", "museum"]`

---
## Your Tasks
1. Implement the Viterbi algorithm.
2. Compute the most likely weather sequence for the 4 days.
3. Print the final most likely hidden-state path.
4. Predict probabilities for Day 5 and 6.

In [1]:
# Define model parameters
states = ["sunny", "rainy"]
observations = ["walk", "shop", "museum"]

pi = {"sunny": 0.6, "rainy": 0.4}

A = {
    "sunny": {"sunny": 0.7, "rainy": 0.3},
    "rainy": {"sunny": 0.4, "rainy": 0.6}
}

B = {
    "sunny": {"walk": 0.6, "shop": 0.3, "museum": 0.1},
    "rainy": {"walk": 0.1, "shop": 0.4, "museum": 0.5}
}

O = ["walk", "shop", "museum", "museum"]

states, O

(['sunny', 'rainy'], ['walk', 'shop', 'museum', 'museum'])

In [12]:
#load libraries
import pandas as pd

# 1. Viterbi algorithm implementation
def viterbi(O, states, pi, A, B):
    # V[t][s] lưu xác suất cao nhất để đi đến trạng thái s tại thời điểm t
    V = [{}]

    # path[s] lưu chuỗi trạng thái tốt nhất dẫn đến trạng thái s
    path = {}

    # Bước khởi tạo (t = 0)
    for s in states:
        # V_0(s) = pi[s] * B[s][O[0]]
        V[0][s] = pi[s] * B[s][O[0]]
        path[s] = [s]

    # Bước đệ quy: tính từ t = 1 đến t = len(O)-1
    for t in range(1, len(O)):
        V.append({})
        new_path = {}

        for j in states:
            # Tìm trạng thái i* ở t-1 cho xác suất: V[t-1][i] * A[i][j] lớn nhất
            (prob, best_state) = max((V[t-1][i] * A[i][j], i) for i in states)

            # Nhân thêm xác suất phát B[j][O[t]]
            V[t][j] = prob * B[j][O[t]]

            # Cập nhật đường đi tốt nhất dẫn tới j
            new_path[j] = path[best_state] + [j]

        path = new_path

    # Bước kết thúc: chọn trạng thái kết thúc có xác suất cao nhất
    final_state = max(states, key=lambda s: V[-1][s])

    return V, final_state, path

# 2. Compute the most likely weather sequence for the 4 days.
V, final_state, path_dict = viterbi(O, states, pi, A, B)

df = pd.DataFrame(V)
df.index = ["Day 1", "Day 2", "Day 3", "Day 4"]
df


,sunny,rainy
Day 1,0.360000,0.040000
Day 2,0.075600,0.043200
Day 3,0.005292,0.012960
Day 4,0.000518,0.003888


In [9]:
# 3. Print final most likely hidden-state sequence
print("Most likely hidden state sequence:", path_dict[final_state])

Most likely hidden state sequence: ['sunny', 'rainy', 'rainy', 'rainy']


In [16]:
# 4. Predict forward state distribution for Day 5 and Day 6

import numpy as np

# Chuyển ma trận A sang dạng numpy
A_mat = np.array([
    [A["rainy"]["rainy"], A["rainy"]["sunny"]],
    [A["sunny"]["rainy"], A["sunny"]["sunny"]]
])

# Lấy phân phối trạng thái ở ngày 4 từ bảng Viterbi
# Ta chuẩn hóa xác suất ở ngày cuối để thành phân phối hợp lệ
last_day = V[-1]
posterior_day4 = np.array([last_day["rainy"], last_day["sunny"]], dtype=float)
posterior_day4 = posterior_day4 / posterior_day4.sum()

# Dự đoán ngày 5
posterior_day5 = posterior_day4 @ A_mat
print("Day 5 state probabilities (sunny, rainy):", posterior_day5)

# Dự đoán ngày 6
posterior_day6 = posterior_day5 @ A_mat
print("Day 6 state probabilities (sunny, rainy):", posterior_day6)

Day 5 state probabilities (sunny, rainy): [0.56470588 0.43529412]
Day 6 state probabilities (sunny, rainy): [0.46941176 0.53058824]
